In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e1/sample_submission.csv
/kaggle/input/playground-series-s6e1/train.csv
/kaggle/input/playground-series-s6e1/test.csv
/kaggle/input/exam-score-prediction-dataset/Exam_Score_Prediction.csv


In [2]:
class CONFIG:
    INPUT_DIR = '/kaggle/input/playground-series-s6e1'

    SEED = 42
    FOLDS = 5
    TARGET = 'exam_score'

config = CONFIG()

In [3]:
train = pd.read_csv(f'{config.INPUT_DIR}/train.csv')
test = pd.read_csv(f'{config.INPUT_DIR}/test.csv')

train_org = pd.read_csv('/kaggle/input/exam-score-prediction-dataset/Exam_Score_Prediction.csv')
submission = pd.read_csv(f'{config.INPUT_DIR}/sample_submission.csv')

train['source'] = 'train'
test['source'] = 'test'
train_org['source'] = 'original'

cols = train.columns
cols = [col for col in cols if col not in ['id', 'student_id']]

test[config.TARGET] = np.nan

train = train[cols].copy()
test = test[cols].copy()
train_org = train_org[cols].copy()

combine = pd.concat([train_org, train, test], axis=0, ignore_index=True)


In [4]:
FEATURES = [col for col in cols if col not in [config.TARGET, 'source']]
CATS = combine[FEATURES].select_dtypes(include='object').columns.to_list()
CATS = [col for col in CATS if col not in ['source']]
NUMS = combine[FEATURES].select_dtypes(include=['int64', 'float64']).columns.to_list()

for df in [combine]:
    for col in NUMS:
        if df[col].dtype=='int64':
            df[col] = df[col].astype('int32')
        else:
            df[col] = df[col].astype('float32')

combine[CATS] = combine[CATS].astype('category')

In [5]:
def preprocess(df):
    df_temp = df.copy()
    eps = 1e-5

    # ================
    # BASIC POLYS
    # ================
    df_temp['study_hours_squared'] = df_temp['study_hours'] ** 2
    df_temp['study_hours_cubed'] = df_temp['study_hours'] ** 3
    df_temp['class_attendance_squared'] = df_temp['class_attendance'] ** 2
    df_temp['sleep_hours_squared'] = df_temp['sleep_hours'] ** 2
    df_temp['age_squared'] = df_temp['age'] ** 2

    # extra polys (tambahan)
    df_temp['study_hours_quartic'] = df_temp['study_hours'] ** 4
    df_temp['class_attendance_cubed'] = df_temp['class_attendance'] ** 3
    df_temp['sleep_hours_cubed'] = df_temp['sleep_hours'] ** 3
    df_temp['age_cubed'] = df_temp['age'] ** 3

    # ================
    # SAFE LOG/SQRT
    # ================
    sh_pos = df_temp['study_hours'].clip(lower=0)
    ca_pos = df_temp['class_attendance'].clip(lower=0)
    sl_pos = df_temp['sleep_hours'].clip(lower=0)
    ag_pos = df_temp['age'].clip(lower=0)

    df_temp['log_study_hours'] = np.log1p(sh_pos)
    df_temp['log_class_attendance'] = np.log1p(ca_pos)
    df_temp['log_sleep_hours'] = np.log1p(sl_pos)

    df_temp['sqrt_study_hours'] = np.sqrt(sh_pos)
    df_temp['sqrt_class_attendance'] = np.sqrt(ca_pos)

    # extra transforms
    df_temp['inv_sleep'] = 1.0 / (sl_pos + 1.0)
    df_temp['inv_study'] = 1.0 / (sh_pos + 1.0)
    df_temp['inv_attendance'] = 1.0 / (ca_pos + 1.0)

    # bounded transforms (stabil)
    df_temp['study_tanh'] = np.tanh(df_temp['study_hours'] / 10.0)
    df_temp['sleep_tanh'] = np.tanh(df_temp['sleep_hours'] / 10.0)
    df_temp['attendance_tanh'] = np.tanh(df_temp['class_attendance'] / 100.0)

    df_temp['study_sigmoid'] = 1.0 / (1.0 + np.exp(-(df_temp['study_hours'] - 5.0)))
    df_temp['sleep_sigmoid'] = 1.0 / (1.0 + np.exp(-(df_temp['sleep_hours'] - 7.0)))
    df_temp['attendance_sigmoid'] = 1.0 / (1.0 + np.exp(-(df_temp['class_attendance'] - 85.0) / 8.0))

    # ================
    # INTERACTIONS (yang kamu punya + tambah)
    # ================
    df_temp['study_hours_times_attendance'] = df_temp['study_hours'] * df_temp['class_attendance']
    df_temp['study_hours_times_sleep'] = df_temp['study_hours'] * df_temp['sleep_hours']
    df_temp['attendance_times_sleep'] = df_temp['class_attendance'] * df_temp['sleep_hours']

    # interactions tambahan
    df_temp['age_times_study_hours'] = df_temp['age'] * df_temp['study_hours']
    df_temp['age_times_attendance'] = df_temp['age'] * df_temp['class_attendance']
    df_temp['age_times_sleep_hours'] = df_temp['age'] * df_temp['sleep_hours']

    # centered interactions (kadang membantu)
    df_temp['study_center_5'] = df_temp['study_hours'] - 5.0
    df_temp['sleep_center_7'] = df_temp['sleep_hours'] - 7.0
    df_temp['att_center_85'] = df_temp['class_attendance'] - 85.0
    df_temp['study_center_sq'] = df_temp['study_center_5'] ** 2
    df_temp['sleep_center_sq'] = df_temp['sleep_center_7'] ** 2
    df_temp['att_center_sq'] = df_temp['att_center_85'] ** 2

    # ================
    # RATIOS (yang kamu punya + tambah)
    # ================
    df_temp['study_hours_over_sleep'] = df_temp['study_hours'] / (df_temp['sleep_hours'] + eps)
    df_temp['attendance_over_sleep'] = df_temp['class_attendance'] / (df_temp['sleep_hours'] + eps)

    # ratios tambahan
    df_temp['attendance_over_study'] = df_temp['class_attendance'] / (df_temp['study_hours'] + eps)
    df_temp['sleep_over_study'] = df_temp['sleep_hours'] / (df_temp['study_hours'] + eps)
    df_temp['study_over_age'] = df_temp['study_hours'] / (df_temp['age'] + eps)
    df_temp['attendance_over_age'] = df_temp['class_attendance'] / (df_temp['age'] + eps)

    # ================
    # CLIPPED + GAPS
    # ================
    df_temp['study_hours_clip'] = df_temp['study_hours'].clip(0, 12)
    df_temp['sleep_hours_clip'] = df_temp['sleep_hours'].clip(0, 12)
    df_temp['attendance_clip'] = df_temp['class_attendance'].clip(0, 100)

    df_temp['sleep_gap_8'] = (df_temp['sleep_hours'] - 8.0).abs()
    df_temp['sleep_gap_7'] = (df_temp['sleep_hours'] - 7.0).abs()
    df_temp['attendance_gap_100'] = (df_temp['class_attendance'] - 100.0).abs()
    df_temp['attendance_gap_90'] = (df_temp['class_attendance'] - 90.0).abs()
    df_temp['study_gap_6'] = (df_temp['study_hours'] - 6.0).abs()
    df_temp['study_gap_8'] = (df_temp['study_hours'] - 8.0).abs()

    # ================
    # BINS (numerik sederhana)
    # ================
    df_temp["age_bin_num"] = pd.cut(df_temp["age"], bins=[0,17,19,21,23,100], labels=[0,1,2,3,4]).astype(float)
    df_temp["study_bin_num"] = pd.cut(df_temp["study_hours"], bins=[-1,2,4,6,8,100], labels=[0,1,2,3,4]).astype(float)
    df_temp["sleep_bin_num"] = pd.cut(df_temp["sleep_hours"], bins=[-1,5,6,7,8,100], labels=[0,1,2,3,4]).astype(float)
    df_temp["attendance_bin_num"] = pd.cut(df_temp["class_attendance"], bins=[-1,60,75,85,95,101], labels=[0,1,2,3,4]).astype(float)

    # ================
    # ORDINAL ENCODING (FIX: moderate)
    # ================
    sleep_quality_map = {'poor': 0, 'average': 1, 'good': 2}
    facility_rating_map = {'low': 0, 'medium': 1, 'high': 2}
    exam_difficulty_map = {'easy': 0, 'moderate': 1, 'hard': 2}

    df_temp['sleep_quality_numeric'] = df_temp['sleep_quality'].map(sleep_quality_map).fillna(1).astype(int)
    df_temp['facility_rating_numeric'] = df_temp['facility_rating'].map(facility_rating_map).fillna(1).astype(int)
    df_temp['exam_difficulty_numeric'] = df_temp['exam_difficulty'].map(exam_difficulty_map).fillna(1).astype(int)

    # interaksi ordinal dengan numerik
    df_temp['study_hours_times_sleep_quality'] = df_temp['study_hours'] * df_temp['sleep_quality_numeric']
    df_temp['attendance_times_facility'] = df_temp['class_attendance'] * df_temp['facility_rating_numeric']
    df_temp['sleep_hours_times_difficulty'] = df_temp['sleep_hours'] * df_temp['exam_difficulty_numeric']

    # ordinal cross
    df_temp['facility_x_sleepq'] = df_temp['facility_rating_numeric'] * df_temp['sleep_quality_numeric']
    df_temp['difficulty_x_facility'] = df_temp['exam_difficulty_numeric'] * df_temp['facility_rating_numeric']
    df_temp['difficulty_x_sleepq'] = df_temp['exam_difficulty_numeric'] * df_temp['sleep_quality_numeric']

    # ================
    # FLAGS (rule-like)
    # ================
    df_temp["high_att_low_sleep"] = ((df_temp["class_attendance"] >= 90) & (df_temp["sleep_hours"] <= 6)).astype(int)
    df_temp["high_att_high_study"] = ((df_temp["class_attendance"] >= 90) & (df_temp["study_hours"] >= 6)).astype(int)
    df_temp["low_att_high_study"] = ((df_temp["class_attendance"] <= 60) & (df_temp["study_hours"] >= 7)).astype(int)
    df_temp["ideal_sleep_flag"] = ((df_temp["sleep_hours"] >= 7) & (df_temp["sleep_hours"] <= 9)).astype(int)
    df_temp["short_sleep_flag"] = (df_temp["sleep_hours"] <= 5.5).astype(int)
    df_temp["high_study_flag"] = (df_temp["study_hours"] >= 7).astype(int)

    # ================
    # COMPOSITE
    # ================
    df_temp['efficiency'] = (df_temp['study_hours'] * df_temp['class_attendance']) / (df_temp['sleep_hours'] + 1)

    df_temp["efficiency2"] = (
        (df_temp["study_hours_clip"] * df_temp["attendance_clip"]) / (df_temp["sleep_hours_clip"] + 1)
    )

    df_temp["weighted_sum"] = (
        0.06 * df_temp["class_attendance"] +
        2.0  * df_temp["study_hours"] +
        1.2  * df_temp["sleep_hours"]
    )

    df_temp["weighted_sum_x_difficulty"] = df_temp["weighted_sum"] * (1.0 + 0.2 * df_temp["exam_difficulty_numeric"])

    # ------------------------
    # list numeric features
    # ------------------------
    numeric_features = [
        # original + extra polys
        'study_hours_squared', 'study_hours_cubed', 'study_hours_quartic',
        'class_attendance_squared', 'class_attendance_cubed',
        'sleep_hours_squared', 'sleep_hours_cubed',
        'age_squared', 'age_cubed',

        # transforms
        'log_study_hours', 'log_class_attendance', 'log_sleep_hours',
        'sqrt_study_hours', 'sqrt_class_attendance',
        'inv_sleep', 'inv_study', 'inv_attendance',
        'study_tanh', 'sleep_tanh', 'attendance_tanh',
        'study_sigmoid', 'sleep_sigmoid', 'attendance_sigmoid',

        # interactions
        'study_hours_times_attendance', 'study_hours_times_sleep', 'attendance_times_sleep',
        'age_times_study_hours', 'age_times_attendance', 'age_times_sleep_hours',
        'study_center_5', 'sleep_center_7', 'att_center_85',
        'study_center_sq', 'sleep_center_sq', 'att_center_sq',

        # ratios
        'study_hours_over_sleep', 'attendance_over_sleep',
        'attendance_over_study', 'sleep_over_study',
        'study_over_age', 'attendance_over_age',

        # clipped + gaps
        'study_hours_clip', 'sleep_hours_clip', 'attendance_clip',
        'sleep_gap_8', 'sleep_gap_7',
        'attendance_gap_100', 'attendance_gap_90',
        'study_gap_6', 'study_gap_8',

        # bins
        'age_bin_num', 'study_bin_num', 'sleep_bin_num', 'attendance_bin_num',

        # ordinal + interactions
        'sleep_quality_numeric', 'facility_rating_numeric', 'exam_difficulty_numeric',
        'study_hours_times_sleep_quality', 'attendance_times_facility', 'sleep_hours_times_difficulty',
        'facility_x_sleepq', 'difficulty_x_facility', 'difficulty_x_sleepq',

        # flags
        'high_att_low_sleep', 'high_att_high_study', 'low_att_high_study',
        'ideal_sleep_flag', 'short_sleep_flag', 'high_study_flag',

        # composite
        'efficiency', 'efficiency2',
        'weighted_sum', 'weighted_sum_x_difficulty'
    ]

    return df_temp, numeric_features

combine, ARTIFICIAL_FEATURES = preprocess(combine)

In [6]:
CATS1 = []

for c in NUMS:
    n = f'{c}_cat'
    for df in [combine]:
        df[n] = df[c].astype('category')
        print(df[n].dtype)

    CATS1.append(n)

for c in CATS1:
    combine[c] = combine[c].cat.codes.astype('int32')  
    combine[c] = combine[c].astype('category')         
print(CATS1)

category
category
category
category
['age_cat', 'study_hours_cat', 'class_attendance_cat', 'sleep_hours_cat']


In [7]:
CATS2 = []
SIZES = {}

for c in CATS+CATS1:
    n = f'{c}_enc'
    for df in [combine]:
        # df[c] = df[c].astype('category')
        df[n], _ = df[c].factorize()
        df[n] = df[n].astype('int32')
        s = df[n].max()+1

    CATS2.append(n)
    SIZES[n] = s

print(CATS2)
print('='*30)
print(SIZES)

['gender_enc', 'course_enc', 'internet_access_enc', 'sleep_quality_enc', 'study_method_enc', 'facility_rating_enc', 'exam_difficulty_enc', 'age_cat_enc', 'study_hours_cat_enc', 'class_attendance_cat_enc', 'sleep_hours_cat_enc']
{'gender_enc': 3, 'course_enc': 7, 'internet_access_enc': 2, 'sleep_quality_enc': 3, 'study_method_enc': 5, 'facility_rating_enc': 3, 'exam_difficulty_enc': 3, 'age_cat_enc': 8, 'study_hours_cat_enc': 794, 'class_attendance_cat_enc': 631, 'sleep_hours_cat_enc': 70}


In [8]:
from itertools import combinations

INTER = []

for i, (col1, col2) in enumerate(combinations(CATS, 2)):
    n = f'{col1}_{col2}'
    if i%5==0: print(i, end='-'*3)
    for df in [combine]:
        combine[n] = (combine[col1].astype(str) + '_' + combine[col2].astype(str)).astype('category')

    INTER.append(n)

print(INTER)
print('='*30)
print('INTER LENGTH :', len(INTER))

0---5---10---15---20---['gender_course', 'gender_internet_access', 'gender_sleep_quality', 'gender_study_method', 'gender_facility_rating', 'gender_exam_difficulty', 'course_internet_access', 'course_sleep_quality', 'course_study_method', 'course_facility_rating', 'course_exam_difficulty', 'internet_access_sleep_quality', 'internet_access_study_method', 'internet_access_facility_rating', 'internet_access_exam_difficulty', 'sleep_quality_study_method', 'sleep_quality_facility_rating', 'sleep_quality_exam_difficulty', 'study_method_facility_rating', 'study_method_exam_difficulty', 'facility_rating_exam_difficulty']
INTER LENGTH : 21


In [9]:
train_idx = combine['source'] == 'train'
test_idx = combine['source'] == 'test'
train_org_idx = combine['source'] == 'original'

train_n = combine[train_idx].reset_index(drop=True).copy()
test_n = combine[test_idx].reset_index(drop=True).copy()
train_org_n = combine[train_org_idx].reset_index(drop=True).copy()

In [10]:
BINS = []
q_list = [10, 20, 30]

for col in NUMS:
    for q in q_list:
        n = f'{col}_bin{q}'

        _, bins = pd.qcut(
            train_org_n[col], q=q, retbins=True, labels=False,
            duplicates='drop'
        )

        train_org_n[n] = pd.cut(train_org_n[col], bins=bins, labels=False, include_lowest=True).astype(np.int8)
        train_n[n] = pd.cut(train_n[col], bins=bins, labels=False, include_lowest=True).astype(np.int8)
        test_n[n] = pd.cut(test_n[col], bins=bins, labels=False, include_lowest=True).astype(np.int8)

        BINS.append(n)

print(f"{len(BINS)} k-bins discretization features created")

12 k-bins discretization features created


In [11]:
TE = []
TE1 = []

for c in CATS+CATS1+BINS+INTER:
    for target in [config.TARGET, 'study_hours']:
        tmp_mean = train_org_n.groupby(c)[target].mean()
        tmp_median = train_org_n.groupby(c)[target].median()
        tmp_std = train_org_n.groupby(c)[target].std()
        tmp_min = train_org_n.groupby(c)[target].min()
        tmp_max = train_org_n.groupby(c)[target].max()
        # tmp_count = train_org_n.groupby(c)[config.TARGET].size()

        n_mean = f'TE_{c}_mean_{target}'
        n_median = f'TE_{c}_median_{target}'
        n_std = f'TE_{c}_std_{target}'
        n_min = f'TE_{c}_min_{target}'
        n_max = f'TE_{c}_max_{target}'

        # n_c = f'TE_{c}_count'
        print(f'{n_mean}, {n_median}, {n_std}, {n_min}, {n_max}', end=' ')
        tmp_mean.name = n_mean
        tmp_median.name = n_median
        tmp_std.name = n_std
        tmp_min.name = n_min
        tmp_max.name = n_max
    
        stats = (pd.concat([tmp_mean, tmp_median, tmp_std, tmp_min, tmp_max], axis=1).reset_index().rename(columns={'index': c}))
        train_org_n = train_org_n.merge(stats, on=c, how='left')
        train_n = train_n.merge(stats, on=c, how='left')
        test_n = test_n.merge(stats, on=c, how='left')

        if target==config.TARGET:
            TE.append(n_mean)
            # TE.append(n_median)
            TE.append(n_std)
            TE.append(n_min)
            TE.append(n_max)

        else:
            TE1.append(n_mean)
            # TE.append(n_median)
            TE1.append(n_std)
            TE1.append(n_min)
            TE1.append(n_max)   
    # TE.append(n_c)

print('\n',)
print(len(TE))
print(len(TE1))

TE_gender_mean_exam_score, TE_gender_median_exam_score, TE_gender_std_exam_score, TE_gender_min_exam_score, TE_gender_max_exam_score TE_gender_mean_study_hours, TE_gender_median_study_hours, TE_gender_std_study_hours, TE_gender_min_study_hours, TE_gender_max_study_hours TE_course_mean_exam_score, TE_course_median_exam_score, TE_course_std_exam_score, TE_course_min_exam_score, TE_course_max_exam_score TE_course_mean_study_hours, TE_course_median_study_hours, TE_course_std_study_hours, TE_course_min_study_hours, TE_course_max_study_hours TE_internet_access_mean_exam_score, TE_internet_access_median_exam_score, TE_internet_access_std_exam_score, TE_internet_access_min_exam_score, TE_internet_access_max_exam_score TE_internet_access_mean_study_hours, TE_internet_access_median_study_hours, TE_internet_access_std_study_hours, TE_internet_access_min_study_hours, TE_internet_access_max_study_hours TE_sleep_quality_mean_exam_score, TE_sleep_quality_median_exam_score, TE_sleep_quality_std_exam_s

In [12]:
def rmse(y, y_val):
    y_array = np.array(y)
    y_val_array = np.array(y_val)
    mse = np.mean((y_array - y_val_array)**2)
    rmse = np.sqrt(mse)
    return rmse

In [13]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

kf = KFold(n_splits=config.FOLDS, shuffle=True, random_state=config.SEED)

oof_pred_lr = np.zeros(len(train_n))
org_pred_lr = np.zeros(len(train_org_n))
test_pred_lr = np.zeros(len(test_n))

fold_rmse_lr = []
lr_models = []
FEATURES_LR = TE + NUMS + CATS2 
X_lr = train_n[FEATURES_LR].copy()
X_org_lr = train_org_n[FEATURES_LR].copy()
test_n_lr = test_n[FEATURES_LR].copy()
X_lr = X_lr.fillna(0)
X_org_lr = X_org_lr.fillna(0)
test_n_lr = test_n_lr.fillna(0)
y = train_n[config.TARGET].copy()
y_org = train_org_n[config.TARGET].copy()
for fold, (train_idx, val_idx) in enumerate(kf.split(X_lr, y), 1):
    print(f'CURRENTly FOLD{fold} BEING PROCESSED')
    X_org_lr_n = X_org_lr.copy()
    X_train, X_val = X_lr.iloc[train_idx], X_lr.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    X_train = pd.concat([X_train, X_org_lr_n], axis=0)
    y_train = pd.concat([y_train, y_org], axis=0)

    # scaler = StandardScaler()
    # X_train[NUMS] = scaler.fit_transform(X_train[NUMS])
    # test_n[NUMS] = scaler.transform(test_n[NUMS])
    # X_org_lr_n[NUMS] = scaler.transform(X_org_lr_n[NUMS])
    print('TRAINING SHAPE :', X_train.shape)
    alphas = np.logspace(-3, 3, 20)
    lr_model = RidgeCV(alphas=alphas, cv=5, scoring='neg_root_mean_squared_error')
    lr_model.fit(X_train, y_train)
    lr_models.append(lr_model)

    lr_val_pred = lr_model.predict(X_val)
    # lr_test_pred = lr_model.predict(test_n_lr)

    oof_pred_lr[val_idx] = lr_val_pred
    org_pred_lr+=(lr_model.predict(X_org_lr)) / config.FOLDS
    test_pred_lr+=(lr_model.predict(test_n_lr)) / config.FOLDS

    rmse_lr = rmse(y_val, lr_val_pred)
    fold_rmse_lr.append(rmse_lr)
    print(f'Fold {fold} RMSE : {rmse_lr}')


print('\n OOF RMSE for LR :', np.mean(fold_rmse_lr))

CURRENTly FOLD1 BEING PROCESSED
TRAINING SHAPE : (524000, 191)
Fold 1 RMSE : 8.828085897559689
CURRENTly FOLD2 BEING PROCESSED
TRAINING SHAPE : (524000, 191)
Fold 2 RMSE : 8.832346213712677
CURRENTly FOLD3 BEING PROCESSED
TRAINING SHAPE : (524000, 191)
Fold 3 RMSE : 8.827924357098993
CURRENTly FOLD4 BEING PROCESSED
TRAINING SHAPE : (524000, 191)
Fold 4 RMSE : 8.839806715258247
CURRENTly FOLD5 BEING PROCESSED
TRAINING SHAPE : (524000, 191)
Fold 5 RMSE : 8.858458909739511

 OOF RMSE for LR : 8.837324418673823


In [14]:
import joblib

joblib.dump(lr_models, 'lr_models.pkl')

joblib.dump(oof_pred_lr, 'oof_pred_lr.pkl')
joblib.dump(org_pred_lr, 'org_pred_lr.pkl')
joblib.dump(test_pred_lr, 'test_pred_lr.pkl')

['test_pred_lr.pkl']